In [ ]:
%cd ..

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [2]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [3]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [ ]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/finance-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "finance_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/finance_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [5]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [6]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [43]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "DIỄN GIẢI":"description",
     "Giá trị":"value"

}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\cong_ty_Financial_ratio_T01_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["source_file"] = filename.split("\\")[-1]

resource_name = "financial_ratio"

df.head()
print(len(df))
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


3
✅ Done: financial_ratio.json created


In [44]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  financial_ratio
financial_ratio
Replace Upload  s3a://vcs-raw/finance-raw/financial_ratio ./tmp/data/financial_ratio/data_financial_ratio_20260318_204550.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/financial_ratio
Uploaded SQL definition


False

In [45]:

COLUMN_DICT_COST  = {
    "Ngày": "report_date",
    "DIỄN GIẢI":"description",
     "Giá trị":"value"

}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\finance-data\data\cong_ty_Financial_ratio_T02_2026.xlsx"

df = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=COLUMN_DICT_COST,
    header_row=0,
    drop_rows=1,
)
df["source_file"] = filename.split("\\")[-1]

resource_name = "financial_ratio"

df.head()
print(len(df))
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


3
✅ Done: financial_ratio.json created


In [46]:

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

Start crawl :  financial_ratio
financial_ratio
Add Upload  s3a://vcs-raw/finance-raw/financial_ratio ./tmp/data/financial_ratio/data_financial_ratio_20260318_204603.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/financial_ratio
Uploaded SQL definition


False

In [ ]:

COLUMN_DICT_SPDV = {
    "Ngày": "date_key",
    "SPDV": "product_service_name",
  "Mã SPDV": "product_service_code",
  "Mã KM phí": "expense_code",
  "Khoản mục SPDV": "product_service_expense_category",
  "Data type": "data_type",
  "Thị trường": "market",
  "Triệu đồng": "amount_million_vnd"
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\cost-2025\chi_phi_spdv_2025"
# resource_name = "product_service_costs"
resource_name = "production_cost_allocation"

df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_SPDV,
    header_row=0,
    drop_rows=1,
)

df.head()

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

In [ ]:
df.head(5)

In [ ]:

COLUMN_DICT_FINANCE_ACCOUNTS_RECEIVABLE = {
  "ngày báo cáo": "report_date",
  "Ngày báo cáo": "report_date",

  "Ngày lập hóa đơn": "invoice_date",
  "ngày lập hóa đơn": "invoice_date",

  "Ngày/Tháng/Năm bắt đầu tính tuổi nợ": "aging_start_date",

  "Nguyên tệ": "original_currency",

  "Nội dung hóa đơn": "invoice_description",

  "Phụ trách KD": "sales_owner",

  "Số Hóa đơn": "invoice_number",
  "Số hóa đơn": "invoice_number",

  "Tên khách hàng BCCS": "customer_name",

  "Tuổi nợ (Tháng)": "aging_months",

  "Tỷ giá": "exchange_rate",

  "VAT": "vat",

  "Mã SAP": "sap_code",

  "Mã giao dich nhận tiền": "payment_transaction_code",
  "Mã giao dịch nhận tiền": "payment_transaction_code",

  "Hợp đồng": "contract_number",

  "Hóa đơn trước thuế": "invoice_amount_before_tax",

  "Giá trị phải thu ban đầu": "initial_receivable_amount",

  "Giá trị hợp đồng": "contract_value",

  "Ghi chú": "note",

  "CÔNG TY": "company",
  "Công ty": "company",

  "Công nợ Đã thu + đã bù trừ công nợ": "collected_and_offset_receivable",

  "Công nợ còn phải thu": "outstanding_receivable",

  "Chưa đến hạn thanh toán": "not_due_yet",

  "Chậm trên 1 năm": "overdue_over_12_months",
  "Chậm trên 1 năm ": "overdue_over_12_months",

  "Chậm 6 tháng -1 năm": "overdue_6_to_12_months",
  "Chậm 6 tháng - 1 năm": "overdue_6_to_12_months",

  "Chậm 3 tháng -6 tháng": "overdue_3_to_6_months",
  "Chậm 3 tháng - 6 tháng": "overdue_3_to_6_months",

  "Chậm < 3 tháng": "overdue_under_3_months",
  "Chậm <3 tháng": "overdue_under_3_months",

  "Chậm < 1 tháng": "overdue_under_1_month",
  "Chậm <1 tháng": "overdue_under_1_month"
}


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\cost-2025\cong_no_2025"
# resource_name = "accounts_receivable"
resource_name = "debt_report"

df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_FINANCE_ACCOUNTS_RECEIVABLE,
    header_row=0,
    drop_rows=1,
)

df.head()

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

In [6]:
COLUMN_DICT_ACTUAL_REVENUE  = {
  "share": "share",
  "chia sẻ": "share_2",
  "date": "report_date",
  "Ngày xuất hóa đơn": "invoice_issue_date",
  "month": "month",

  "revenue": "revenue",
  "Doanh thu": "revenue_2",

  "customer": "customer_name",
  "Phân loại KH": "customer_segment",
  "Nhóm khách hàng": "customer_group",

  "old_category": "old_category",
  "category_code": "category_code",
  "Nhóm doanh thu": "revenue_group",

  "Nội dung đề nghị TT(Theo chứng từ gốc)": "payment_request_description",

  "Segment3": "segment_level_3",
  "producttype": "product_type",
  "group_spdv": "product_service_group",
  "territory": "territory",

  "is_SOC": "is_soc",
  "VAT": "vat_amount",
  "Sum of Tiền hàng (đã bao gồm VAT)": "gross_amount_including_vat",

  "AM": "account_manager",
  "department": "department",
  "presale": "presale_owner",

  "channel": "sales_channel",
  "payment_stage": "payment_stage",

  "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",
  "Trung tâm DT": "revenue_allocated_unit"
}

#filename = r"C:\Users\namtv40\Documents\AI Chatbot\csv-revenue-finance-pbi_2025\actual-revenue"
filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\csv-revenue-finance-pbi- 2025\csv-revenue-finance-pbi- 2025\positive"
resource_name = "actual_revenue"
df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_ACTUAL_REVENUE,
    header_row=0,
    drop_rows=1,
)

df.head()

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: actual_revenue.json created
Start crawl :  actual_revenue
actual_revenue
Replace Upload  s3a://vcs-raw/finance-raw/actual_revenue ./tmp/data/actual_revenue/data_actual_revenue_20260314_174034.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/actual_revenue
Uploaded SQL definition


False

In [7]:
COLUMN_DICT_ESTIMATED_REVENUE  = {
  "share": "share",
  "chia sẻ": "share_2",

  "date": "report_date",
  "month": "month",
  "Ngày xuất hóa đơn": "invoice_issue_date",

  "revenue": "revenue",
  "Doanh thu": "revenue_2",

  "customer": "customer_name",
  "Phân loại KH": "customer_segment",
  "Nhóm khách hàng": "customer_group",

  "old_category": "old_category",
  "category_code": "category_code",
  "Nhóm doanh thu": "revenue_group",

  "Segment3": "segment_level_3",

  "producttype": "product_type",
  "group_spdv": "product_service_group",
  "territory": "territory",

  "is_SOC": "is_soc",
  "VAT": "vat_amount",
  "Sum of Tiền hàng (đã bao gồm VAT)": "gross_amount_including_vat",

  "AM": "account_manager",
  "department": "department",
  "presale": "presale_owner",

  "channel": "sales_channel",
  "payment_stage": "payment_stage",

  "Nội dung đề nghị TT(Theo chứng từ gốc)": "payment_request_description",
  "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",
  "Trung tâm DT": "revenue_allocated_unit",
}

# filename = r"C:\Users\namtv40\Documents\AI Chatbot\csv-revenue-finance-pbi_2025\estimated-revenue"
filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\csv-revenue-finance-pbi- 2025\csv-revenue-finance-pbi- 2025\negative"
resource_name = "provisional_revenue"
df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_ESTIMATED_REVENUE,
    header_row=0,
    drop_rows=1,
)

df.head()

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: provisional_revenue.json created
Start crawl :  provisional_revenue
provisional_revenue
Replace Upload  s3a://vcs-raw/finance-raw/provisional_revenue ./tmp/data/provisional_revenue/data_provisional_revenue_20260314_174129.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/provisional_revenue
Uploaded SQL definition


False

In [ ]:
COLUMN_DICT_PLAN_FIN  = {
  "segment":"segment"
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\taichinh\raw"
resource_name = "finance_plan"
df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_PLAN_FIN,
    header_row=0,
    drop_rows=1,
)
df["subgroup"] = ""
df.head()

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

In [ ]:
COLUMN_DICT_PLAN_FIN = {
    "Tên nhóm KH": "segment",
    "Phân nhóm KH": "subgroup",
    "Tập đoàn": "plan_viettel_group",
    "Must": "plan_must",
    "Nice": "plan_nice",
    "Tháng": "plan_month",
    "Năm": "plan_year",
    "Ngày tháng": "plan_date",
    "Đơn vị": "currency_code",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx"
resource_name = "finance_plan"
df = pd.read_excel(filename, dtype=str)

# Remove whitespace ở header nếu có
df.columns = df.columns.str.strip()
# Rename
df = df.rename(columns=COLUMN_DICT_PLAN_FIN)

df["source_file"] = "plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx"
df["company"] = "VCS"
df["subgroup"] = df["subgroup"].fillna("")


print(df.columns)
print(df.dtypes)
df.head()


In [ ]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

In [8]:
filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\plan_revenue\RAW_Dieu chinh Tach KH tung thang_N2025.xlsx"

import pandas as pd
import re

# ==============================
# 1️⃣ READ FILE
# ==============================
df = pd.read_excel(filename,sheet_name="KH N2025 (theo nhóm SPDV)", header=[0,1,2])

# Flatten header
df.columns = [
    "_".join([str(x).strip() for x in col if pd.notna(x)])
    for col in df.columns
]

# Rename 3 cột đầu
df = df.rename(columns={
    df.columns[0]: "tt",
    df.columns[1]: "ma_spdv",
    df.columns[2]: "ten_spdv",
})

# ==============================
# 2️⃣ IDENTIFY GROUP ROW
# ==============================
def is_roman(val):
    if pd.isna(val):
        return False
    return bool(re.fullmatch(r"[IVXLCDM]+", str(val).strip().upper()))

df["is_group"] = (
    df["ma_spdv"].isna() |
    df["ma_spdv"].astype(str).str.strip().eq("") |
    df["tt"].apply(is_roman)
)

# ==============================
# 3️⃣ CREATE PRODUCT GROUP
# ==============================
df["product_group"] = df["ten_spdv"].where(df["is_group"])
df["product_group"] = df["product_group"].ffill()

# ==============================
# 4️⃣ REMOVE GROUP ROWS
# ==============================
df_detail = df[~df["is_group"]].copy()

# Rename columns
df_detail = df_detail.rename(columns={
    "tt": "row_no",
    "ma_spdv": "product_code",
    "ten_spdv": "product_name",
})

# ==============================
# 5️⃣ MELT MONTH COLUMNS
# ==============================
value_cols = [
    c for c in df_detail.columns
    if "KH T" in c and ("MUST" in c.upper() or "NICE" in c.upper())
]

df_long = df_detail.melt(
    id_vars=["row_no", "product_code", "product_name", "product_group"],
    value_vars=value_cols,
    var_name="raw_column",
    value_name="target_value"
)

# ==============================
# 6️⃣ PARSE PERIOD & TARGET TYPE
# ==============================
def parse_column(col):
    # Extract month/year
    m = re.search(r"T(\d+)/(\d+)", col)
    if not m:
        return pd.Series([None, None, None, None])

    month = int(m.group(1))
    yy = m.group(2)
    year = int("20" + yy) if len(yy) == 2 else int(yy)

    target_type = "must" if "MUST" in col.upper() else "nice"

    period = f"{year}-{month:02d}"

    return pd.Series([period, year, month, target_type])

df_long[["period", "year", "month", "target_type"]] = df_long["raw_column"].apply(parse_column)

# ==============================
# 7️⃣ CLEAN VALUE + FILL NaN = 0
# ==============================
df_long["target_value"] = (
    df_long["target_value"]
        .astype(str)
        .str.replace(",", "", regex=False)
)

df_long["target_value"] = (
    pd.to_numeric(df_long["target_value"], errors="coerce")
      .fillna(0)
      .mul(1_000_000)   # nhân 10^6
      .astype("int64")   # BIGINT
)

# ==============================
# 8️⃣ FINAL CLEANUP
# ==============================
df_long = df_long.drop(columns=["raw_column"])

df_long = df_long.sort_values(
    ["product_group", "product_code", "year", "month", "target_type"]
)
df_long["currency_code"] = "VND"
import pandas as pd

df_long["snapshot_at"] = pd.Timestamp.now(tz="Asia/Ho_Chi_Minh").strftime("%Y-%m-%d %H:%M:%S")
df_long["snapshot_version"] = 1
df_long = df_long.reset_index(drop=True)

# ==============================
# 9️⃣ RESULT
# ==============================
# Pivot target_type thành cột
df_wide = (
    df_long
        .pivot_table(
            index=[
                "row_no",
                "product_group",
                "product_code",
                "product_name",
                "year",
                "month",
                "period",
                "currency_code",
                "snapshot_at",
                "snapshot_version",
            ],
            columns="target_type",
            values="target_value",
            aggfunc="sum",
            fill_value=0   # Quan trọng
        )
        .reset_index()
)

df_wide.columns.name = None

# Đảm bảo có đủ 2 cột
for col in ["must", "nice"]:
    if col not in df_wide.columns:
        df_wide[col] = 0

df_wide["must"] = df_wide["must"].astype("int64")
df_wide["nice"] = df_wide["nice"].astype("int64")

# Sắp xếp
df_wide = df_wide.sort_values(
    ["product_group", "product_code", "year", "month"]
)

df_wide.head(20)


,row_no,product_group,product_code,product_name,year,month,period,currency_code,snapshot_at,snapshot_version,must,nice
0,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,1,2025-01,VND,2026-03-20 11:47:18,1,10967036264,12972906453
1,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,2,2025-02,VND,2026-03-20 11:47:18,1,11478834658,13554673435
2,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,3,2025-03,VND,2026-03-20 11:47:18,1,21081028689,24838547180
3,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,4,2025-04,VND,2026-03-20 11:47:18,1,8646618351,9581995114
4,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,5,2025-05,VND,2026-03-20 11:47:18,1,12121405196,13395154438
5,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,6,2025-06,VND,2026-03-20 11:47:18,1,21925147421,25713911076
6,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,7,2025-07,VND,2026-03-20 11:47:18,1,11606564179,12146342449
7,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,8,2025-08,VND,2026-03-20 11:47:18,1,14283955956,14997575481
8,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,9,2025-09,VND,2026-03-20 11:47:18,1,21155561740,24196563369
9,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,10,2025-10,VND,2026-03-20 11:47:18,1,12798937900,19701470540


In [9]:
print("Số tháng:", df_long["month"].nunique())
print("Danh sách tháng:", sorted(df_long["month"].unique()))

Số tháng: 12
Danh sách tháng: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


In [10]:
# 6. Chuyển sang JSON

resource_name = "product_revenue_plan"
records = df_wide.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: product_revenue_plan.json created
Start crawl :  product_revenue_plan
product_revenue_plan
Replace Upload  s3a://vcs-raw/finance-raw/product_revenue_plan ./tmp/data/product_revenue_plan/data_product_revenue_plan_20260320_114719.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/product_revenue_plan
Uploaded SQL definition


False

In [6]:
# 6. Chuyển sang JSON
df_product = pd.read_excel(r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\dim_product_category_code.xlsx", dtype=str)
df_product["source_file"] = "dim_product_category_code.xlsx"
resource_name = "dim_product_category_code"
records = df_product.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: dim_product_category_code.json created
Start crawl :  dim_product_category_code
dim_product_category_code
Replace Upload  s3a://vcs-raw/finance-raw/dim_product_category_code ./tmp/data/dim_product_category_code/data_dim_product_category_code_20260317_155212.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/dim_product_category_code
Uploaded SQL definition


False

In [67]:

REVENUE_MAPPING = {
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice info =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",  # HD / TT
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",

    # ===== Revenue recognition =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "Mã SPDV": "service_code",

    # ===== Customer / Market =====
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",  # internal / external / international / market
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Khách hàng": "customer_name",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "AM hiện tại": "current_account_manager",
    "AM hiện tại ": "current_account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",  # new / existing
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region =====
    "Bắc/Nam/Thị trường": "region_group",
    "Nam/ Bắc": "region",   # ⚠️ duplicate semantic

    # ===== Flags =====
    "Vvip": "is_vvip_customer",
    "HĐ khung": "is_master_contract",
    "HĐ khung ":"is_master_contract",

    # ===== Notes =====
    "Ghi chú": "note",
}

REVENUE_MAPPING_V2 = {
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    " Doanh thu ": "revenue_amount",
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    " Doanh thu cuối ": "final_revenue_amount",
    "Doanh thu cuối": "final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",

    # ===== Revenue behavior =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "Mã SPDV": "service_code",

    # ===== Customer =====
    "Khách hàng": "customer_name",
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Market / Scope =====
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region (FIX conflict) =====
    "Bắc/Nam/Thị trường": "market_scope",
    "Nam/ Bắc": "region",

    # ===== Note =====
    "Ghi chú": "note",
}

REVENUE_MAPPING = REVENUE_MAPPING | REVENUE_MAPPING_V2


filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 1_2026\Doanh_thu_cong_ty_t1_2026.xlsx"

df1 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Doanh_thu_cong_ty_t1_2026",
    mapping=REVENUE_MAPPING,
    header_row=0,
    drop_rows=1,
)
df1["source_file"] = filename.split("\\")[-1]

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 2_2026\Doanh_thu_cong_ty_t2_2026.xlsx"

df2 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Doanh_thu_cong_ty_t2_2026",
    mapping=REVENUE_MAPPING,
    header_row=0,
    drop_rows=1,
)
df2["source_file"] = filename.split("\\")[-1]
df = pd.concat([df1, df2], axis=0)

# resource_name = "costs"
resource_name = "sales_revenue"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


356
✅ Done: sales_revenue.json created


In [66]:
df.columns

Index(['invoice_date', 'invoice_month', 'invoice_type', 'billing_description',
       'revenue_amount', 'revenue_share_amount', 'final_revenue_amount',
       'vat_amount', 'gross_amount', 'revenue_recognition_type',
       'service_category', 'customer_scope', 'customer_type',
       'customer_channel', 'customer_name', 'customer_segment', 'department',
       'customer_group', 'account_manager', 'presale', 'service_name',
       'revenue_type', 'soc_type', 'mss_shared_revenue', 'market_scope',
       'service_code', 'region', 'note', 'current_account_manager',
       'is_vvip_customer', 'is_master_contract', 'source_file'],
      dtype='object')

In [68]:
resource_name = "sales_revenue"
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  sales_revenue
sales_revenue
Replace Upload  s3a://vcs-raw/finance-raw/sales_revenue ./tmp/data/sales_revenue/data_sales_revenue_20260318_210625.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None

❌ Lỗi request khi gọi https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove: 403 Client Error: Forbidden for url: https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None

❌ Lỗi request khi gọi https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove: 403 Client Error: Forbidden for url: https://data

False

In [73]:
SHARED_REVENUE_MAPPING_V3 = {
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",

    # ===== Revenue behavior =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "Mã SPDV": "service_code",

    # ===== Customer =====
    "Khách hàng": "customer_name",
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Market =====
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region (fix conflict) =====
    "Bắc/Nam/Thị trường": "market_scope",
    "Nam/ Bắc": "region",

    # ===== Notes (merge) =====
    "Ghi chú": "note",
    "Note": "note",   # merge vào cùng field
}

SHARED_REVENUE_MAPPING_V4 = {
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",

    # ===== Revenue behavior =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product / Service =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "Mã SPDV": "service_code",

    # ===== Customer =====
    "Khách hàng": "customer_name",
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Market =====
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "AM hiện tại": "current_account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region (resolve conflict) =====
    "Bắc/Nam/Thị trường": "market_scope",
    "Nam/ Bắc": "region",

    # ===== Flags =====
    "Vvip": "is_vvip_customer",
    "HĐ khung": "is_master_contract",

    # ===== Notes (merge) =====
    "Ghi chú": "note",
    "Note": "note",
}

SHARED_REVENUE_MAPPING_V3 = SHARED_REVENUE_MAPPING_V3 | SHARED_REVENUE_MAPPING_V4

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 1_2026\Doanh_thu_cong_ty_t1_2026_shared_all.xlsx"

df1 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=SHARED_REVENUE_MAPPING_V3,
    header_row=0,
    drop_rows=1,
)
df1["source_file"] = filename.split("\\")[-1]

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 2_2026\Doanh_thu_cong_ty_t2_2026_shared_all.xlsx"

df2 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=SHARED_REVENUE_MAPPING_V3,
    header_row=0,
    drop_rows=1,
)
df2["source_file"] = filename.split("\\")[-1]
df = pd.concat([df1, df2], axis=0)

resource_name = "sales_revenue_share_all"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


791
✅ Done: sales_revenue_share_all.json created


In [74]:
resource_name = "sales_revenue_share_all"
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  sales_revenue_share_all
sales_revenue_share_all
Replace Upload  s3a://vcs-raw/finance-raw/sales_revenue_share_all ./tmp/data/sales_revenue_share_all/data_sales_revenue_share_all_20260318_211258.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/sales_revenue_share_all
Uploaded SQL definition


False

In [76]:
SHARED_SOC_REVENUE_MAPPING_FINAL = {
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",

    # ===== Revenue behavior =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "Mã SPDV": "service_code",

    # ===== Customer =====
    "Khách hàng": "customer_name",
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Market =====
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "AM hiện tại": "current_account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region =====
    "Nam/ Bắc": "region",
    "Bắc/Nam/Thị trường": "market_scope",

    # ===== Flags =====
    "Vvip": "is_vvip_customer",
    "HĐ khung": "is_master_contract",

    # ===== Note =====
    "Ghi chú": "note",
    "Note": "note",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 1_2026\Doanh_thu_cong_ty_t1_2026_SOC_chia_se_cho_SPDV.xlsx"

df1 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=SHARED_SOC_REVENUE_MAPPING_FINAL,
    header_row=0,
    drop_rows=1,
)
df1["source_file"] = filename.split("\\")[-1]

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 2_2026\Doanh_thu_cong_ty_t2_2026_SOC_chia_se_cho_SPDV.xlsx"

df2 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=SHARED_SOC_REVENUE_MAPPING_FINAL,
    header_row=0,
    drop_rows=1,
)
df2["source_file"] = filename.split("\\")[-1]
df = pd.concat([df1, df2], axis=0)

# resource_name = "costs"
resource_name = "sales_revenue_share_soc_product_category"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


704
✅ Done: sales_revenue_share_soc_product_category.json created


In [77]:
resource_name = "sales_revenue_share_soc_product_category"
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  sales_revenue_share_soc_product_category
sales_revenue_share_soc_product_category
Replace Upload  s3a://vcs-raw/finance-raw/sales_revenue_share_soc_product_category ./tmp/data/sales_revenue_share_soc_product_category/data_sales_revenue_share_soc_product_category_20260318_211730.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/sales_revenue_share_soc_product_category
Uploaded SQL definition


False

In [13]:
CUSTOMER_PRODUCT_MAPPING = {
    # ===== Customer group =====
    "Tên nhóm KH": "customer_group_name",
    "Phân nhóm KH": "customer_subgroup_name",
    "Nhóm": "group_name",
    "Phân nhóm": "subgroup_name",

    # ===== Product / Service =====
    "Mã SPDV": "service_code",
    "Sản phẩm": "service_name",
    "Đơn vị tính": "amount_unit",

    # ===== Priority / Importance =====
    "Must": "priority_must",
    "Should": "priority_should",
    "Nice": "priority_nice",

    # ===== Time =====
    "Tháng": "report_month",
    "Năm": "report_year",
    "Ngày tháng": "report_date",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Plan_SPDV_2026.xlsx"

df2 = read_excel_and_normalize_columns(
      filename,
    sheet_name="product_revenue_plan",
    mapping=CUSTOMER_PRODUCT_MAPPING,
    header_row=0,
    drop_rows=1,
)
df2["source_file"] = filename.split("\\")[-1]
df = df2.fillna("")

resource_name = "product_revenue_plan_2026"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


1584
✅ Done: product_revenue_plan_2026.json created


In [14]:
resource_name = "product_revenue_plan_2026"
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  product_revenue_plan_2026
product_revenue_plan_2026
Replace Upload  s3a://vcs-raw/finance-raw/product_revenue_plan_2026 ./tmp/data/product_revenue_plan_2026/data_product_revenue_plan_2026_20260320_114929.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/product_revenue_plan_2026
Uploaded SQL definition


False